In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

data = pd.read_csv(r'C:\Users\PAARTH\Documents\Neural-Network\Data\train.csv')
data.head()

In [ ]:
data = np.array(data)  #Making it a numpay array for future
m, n = data.shape #m-> rows(examples) n->columns(amount of features+1)
np.random.shuffle(data) #shuffling ensures random training samples

#validation set
data_dev = data[0:1000].T #1st 1000 examples
Y_dev = data_dev[0] #labels
X_dev = data_dev[1:n] #image pixels

#training set
data_train = data[1000:m].T #(785, 41000)
Y_train = data_train[0] #(41000, )
X_train = data_train[1:n] #(784, 41000)    784 features, 41000 training images

X_train = X_train / 255 #normalizing the pixel values
X_dev = X_dev / 255

In [ ]:
#initializing weights and biases

def init_params():
    W1 = np.random.rand(10,784) - 0.5    #10 neurons, 784 inputs
    b1 = np.random.rand(10,1) - 0.5      #one bias per neuron
    W2 = np.random.rand(10,10) - 0.5     #10 output neurons, 10 hidden neurons
    b2 = np.random.rand(10,1) - 0.5      #one bias per output neuron
    return W1,b1,W2,b2

#Why don't we just make the weights zero?
#why the -0.5?

In [ ]:
def ReLU(Z):                #why do we need ReLU?
    return np.maximum(0,Z)  

def softmax(Z):            #why do we require softmax? Whats the formula?
    expZ = np.exp(Z - np.max(Z, axis=0, keepdims=True))  #subtracting max prevents numerical overflow
    return expZ / np.sum(expZ, axis=0, keepdims=True) #normalizes values so each column sums to 1

In [ ]:
def forward_prop(W1, b1, W2, b2, X):

    Z1 = W1@X + b1
    A1 = ReLU(Z1)

    Z2 = W2@A1 + b2
    A2 = softmax(Z2)

    return Z1, A1, Z2, A2

In [ ]:
def one_hot(Y):

    one_hot_Y = np.zeros((Y.size,Y.max()+1))
    one_hot_Y[np.arange(Y.size),Y] = 1
    one_hot_Y = one_hot_Y.T

    return one_hot_Y #(10,m)

#7 → [0 0 0 0 0 0 0 1 0 0]
#2 → [0 0 1 0 0 0 0 0 0 0]
#1 → [0 1 0 0 0 0 0 0 0 0]
#9 → [0 0 0 0 0 0 0 0 0 1] (we transpose this afterwards)


In [ ]:
def backward_prop(Z1, A1, Z2, A2, W2, X, Y):
    m = Y.size
    one_hot_Y = one_hot(Y)

    dZ2 = A2- one_hot_Y
    dW2 = 1/m * dZ2 @ A1.T     
    db2 = 1/m * np.sum(dZ2, axis=1, keepdims=True)

    dZ1 = W2.T @ dZ2 * (Z1 > 0 )      #(Z1 >0 ) -> Derivative of ReLU
    dW1 = 1/m * dZ1 @ X.T
    db1 =  1/m * np.sum(dZ1, axis=1, keepdims=True)

    return dW1, db1, dW2, db2

In [ ]:
def update_params(W1,b1,W2,b2,dW1,db1,dW2,db2,alpha):

    W1 = W1 - alpha*dW1   #alpha -> learning rate
    b1 = b1 - alpha*db1
    
    W2 = W2 - alpha*dW2
    b2 = b2 - alpha*db2

    return W1,b1,W2,b2

In [ ]:
def get_predictions(A2):
    return np.argmax(A2,0)  #highest prob digit

def get_accuracy(predictions,Y): 
    print(predictions,Y)
    return np.sum(predictions==Y)/Y.size   # % of correct predictions

In [ ]:
def gradient_descent(X,Y,iterations, alpha):      #training

    W1, b1, W2, b2 = init_params()

    for i in range(iterations):

        Z1,A1,Z2,A2 = forward_prop(W1,b1,W2,b2,X)

        dW1,db1,dW2,db2 = backward_prop(Z1,A1,Z2,A2,W2,X,Y)

        W1,b1,W2,b2 = update_params(W1,b1,W2,b2,dW1,db1,dW2,db2,alpha)


        if i%25 == 0:

            print("Iteration: ",i)

            predictions = get_predictions(A2)

            print("Accuracy:",get_accuracy(predictions,Y))
            
    return W1,b1,W2,b2

In [ ]:
W1, b1, W2, b2 = gradient_descent(X_train,Y_train,1000,0.1)

In [ ]:
index = 12


current_image = X_train[:, index].reshape((28,28))

plt.gray()
plt.imshow(current_image)
plt.show()

# run forward pass again
Z1, A1, Z2, A2 = forward_prop(W1, b1, W2, b2, X_train[:, index:index+1])

prediction = get_predictions(A2)

print("Prediction:", prediction)
print("Label:", Y_train[index])